In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Biblioteca de circuitos

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>
O SDK do Qiskit inclui uma biblioteca de circuitos populares para usar como blocos de construção nos seus próprios programas. Usar circuitos pré-definidos economiza tempo de pesquisa, escrita de código e depuração. A biblioteca inclui circuitos populares em computação quântica, circuitos difíceis de simular classicamente e circuitos úteis para benchmarking de hardware quântico.

Esta página lista as diferentes categorias de circuitos que a biblioteca fornece. Para uma lista completa de circuitos, consulte a [documentação da API da biblioteca de circuitos](https://docs.quantum.ibm.com/api/qiskit/circuit_library).
## Gates padrão
A biblioteca de circuitos também inclui gates quânticos padrão. Alguns são gates mais fundamentais (como o `UGate`), e outros são gates multi-qubit que geralmente precisam ser construídos a partir de gates de um e dois qubits. Para adicionar gates importados ao seu circuito, use o método `append`; o primeiro argumento é o gate, e o próximo argumento é uma lista de qubits aos quais aplicar o gate.

Por exemplo, o seguinte bloco de código cria um circuito com um gate de Hadamard e um gate multi-controlled-X.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import HGate, MCXGate

mcx_gate = MCXGate(3)
hadamard_gate = HGate()

qc = QuantumCircuit(4)
qc.append(hadamard_gate, [0])
qc.append(mcx_gate, [0, 1, 2, 3])
qc.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/a846a845-7ac5-4c92-b124-d2b90a773ba2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/a846a845-7ac5-4c92-b124-d2b90a773ba2-0.svg)

Consulte [Standard gates](https://docs.quantum.ibm.com/api/qiskit/circuit_library#standard-gates) na documentação da API da biblioteca de circuitos.

<CodeAssistantAdmonition tagLine="Not sure what your gate's called? Try asking Qiskit Code Assistant." />

## Circuitos N-local

Esses circuitos alternam camadas de gates de rotação de qubit único com camadas de gates de emaranhamento multi-qubit.

Essa família de circuitos é popular em algoritmos quânticos variacionais porque pode produzir uma ampla gama de estados quânticos. Algoritmos variacionais ajustam os parâmetros dos gates para encontrar estados que possuem certas propriedades (como estados que representam uma boa solução para um problema de otimização). Para esse propósito, muitos circuitos na biblioteca são **parametrizados**, o que significa que você pode defini-los sem valores fixos.

O seguinte bloco de código importa um circuito `n_local`, no qual os gates de emaranhamento são gates de dois qubits. Esse circuito intercala blocos de gates de qubit único parametrizados, seguidos por blocos de emaranhamento de gates de dois qubits. O código a seguir cria um circuito de três qubits, com gates RX de qubit único e gates CZ de dois qubits.

In [2]:
from qiskit.circuit.library import n_local

two_local = n_local(3, "rx", "cz")
two_local.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/3ccaeb1b-03c6-4dfa-9000-e48db2516303-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/3ccaeb1b-03c6-4dfa-9000-e48db2516303-0.svg)

Você pode obter um objeto semelhante a uma lista dos parâmetros do circuito a partir do atributo `parameters`.

In [3]:
two_local.parameters

ParameterView([ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), ParameterVectorElement(θ[3]), ParameterVectorElement(θ[4]), ParameterVectorElement(θ[5]), ParameterVectorElement(θ[6]), ParameterVectorElement(θ[7]), ParameterVectorElement(θ[8]), ParameterVectorElement(θ[9]), ParameterVectorElement(θ[10]), ParameterVectorElement(θ[11])])

You can also use this to assign these parameters to real values using a dictionary of the form `{ Parameter: number }`. To demonstrate, the following code cell assigns each parameter in the circuit to `0`.

In [4]:
bound_circuit = two_local.assign_parameters(
    {p: 0 for p in two_local.parameters}
)
bound_circuit.decompose().draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/89227b48-12b2-4b1b-9680-55a7fce88a2b-0.svg" alt="Output of the previous code cell" />

Você também pode usar isso para atribuir esses parâmetros a valores reais usando um dicionário no formato `{ Parameter: number }`. Para demonstrar, o seguinte bloco de código atribui cada parâmetro do circuito ao valor `0`.

In [5]:
from qiskit.circuit.library import zz_feature_map

features = [0.2, 0.4, 0.8]
feature_map = zz_feature_map(feature_dimension=len(features))

encoded = feature_map.assign_parameters(features)
encoded.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/cf8b1efc-57b3-4681-8e6a-d5b8406d092d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/89227b48-12b2-4b1b-9680-55a7fce88a2b-0.svg)

Para mais informações, consulte [N-local gates](https://docs.quantum.ibm.com/api/qiskit/circuit_library#n-local-circuits) na documentação da API da biblioteca de circuitos ou faça o [curso de design de algoritmos variacionais](/learning/courses/variational-algorithm-design) no IBM Quantum Learning.

## Circuitos de codificação de dados

Esses circuitos parametrizados codificam dados em estados quânticos para processamento por algoritmos de aprendizado de máquina quântico. Alguns circuitos suportados pelo Qiskit são:
 - Codificação por amplitude, que codifica cada número na amplitude de um estado de base. Isso pode armazenar $2^n$ números em um único estado, mas pode ser custoso de implementar.
 - Codificação por base, que codifica um inteiro $k$ preparando o estado de base correspondente $|k\rangle$.
 - Codificação por ângulo, que define cada número nos dados como um ângulo de rotação em um circuito parametrizado.

A melhor abordagem depende das especificidades da sua aplicação. Nos computadores quânticos atuais, porém, frequentemente usamos circuitos de codificação por ângulo como o `zz_feature_map`.

In [6]:
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp


# Prepare an initial state with a Hadamard on the middle qubit
state = QuantumCircuit(3)
state.h(1)

hamiltonian = SparsePauliOp(["ZZI", "IZZ"])
evolution = PauliEvolutionGate(hamiltonian, time=1)

# Evolve state by appending the evolution gate
state.compose(evolution, inplace=True)

state.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/834794df-86e9-4bea-8efa-5380499e359b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/cf8b1efc-57b3-4681-8e6a-d5b8406d092d-0.svg)

Consulte [Data encoding circuits](https://docs.quantum.ibm.com/api/qiskit/circuit_library#data-encoding-circuits) na documentação da API da biblioteca de circuitos.

## Circuitos de evolução temporal

Esses circuitos simulam um estado quântico evoluindo no tempo. Use circuitos de evolução temporal para investigar efeitos físicos como transferência de calor ou transições de fase em um sistema. Circuitos de evolução temporal também são um bloco de construção fundamental de funções de onda de química (como estados de teste de cluster acoplado unitário) e do algoritmo QAOA que usamos para problemas de otimização.

In [7]:
from qiskit.circuit.library import quantum_volume

quantum_volume(4).draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/9629a507-8191-409e-b895-fd0833c8fcd7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/834794df-86e9-4bea-8efa-5380499e359b-0.svg)

Leia a [documentação da API do `PauliEvolutionGate`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PauliEvolutionGate).

## Circuitos de benchmarking e teoria da complexidade

Circuitos de benchmarking nos dão uma noção de quão bem nosso hardware está realmente funcionando, e circuitos de teoria da complexidade nos ajudam a entender o quão difíceis são os problemas que queremos resolver.

Por exemplo, o benchmark de "volume quântico" mede com qual precisão um computador quântico executa um tipo de circuito quântico aleatório. A pontuação do computador quântico aumenta com o tamanho do circuito que ele consegue executar de forma confiável. Isso leva em conta todos os aspectos do computador, incluindo contagem de qubits, fidelidade de instruções, conectividade de qubits e a pilha de software que transpila e pós-processa os resultados. Leia mais sobre volume quântico no [artigo original sobre volume quântico](https://arxiv.org/abs/1811.12926).

O código a seguir mostra um exemplo de um circuito de volume quântico construído no Qiskit que roda em quatro qubits (os blocos `unitary` são gates de dois qubits aleatorizados).

In [8]:
from qiskit.circuit.library import FullAdderGate
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

adder = FullAdderGate(3)  # Adder of 3-bit numbers

# Create the number A=2
reg_a = QuantumRegister(3, "a")
number_a = QuantumCircuit(reg_a)
number_a.initialize(2)  # Number 2; |010>

# Create the number B=3
reg_b = QuantumRegister(3, "b")
number_b = QuantumCircuit(reg_b)
number_b.initialize(3)  # Number 3; |011>

# Create a circuit to hold everything, including a classical register for
# the result
qregs = [
    QuantumRegister(1, "cin"),
    QuantumRegister(3, "a"),
    QuantumRegister(3, "b"),
    QuantumRegister(1, "cout"),
]
reg_result = ClassicalRegister(3)
circuit = QuantumCircuit(*qregs, reg_result)

# Compose number initializers with the adder. Adder stores the result to
# register B, so we'll measure those qubits.
circuit = (
    circuit.compose(number_a, qubits=reg_a)
    .compose(number_b, qubits=reg_b)
    .compose(adder)
)
circuit.measure(reg_b, reg_result)
circuit.draw("mpl")

<Image src="../docs/images/guides/circuit-library/extracted-outputs/77555a5a-a81c-47b8-a9ae-3015d84adcf5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/9629a507-8191-409e-b895-fd0833c8fcd7-0.svg)

A biblioteca de circuitos também inclui circuitos considerados difíceis de simular classicamente, como os circuitos de polinômio quântico instantâneo (iqp). Esses circuitos intercalam certos gates diagonais (na base computacional) entre blocos de gates de Hadamard.

Outros circuitos incluem `grover_operator` para uso no algoritmo de Grover, e o circuito `fourier_checking` para o problema de verificação de Fourier. Veja esses circuitos em [Particular quantum circuits](https://docs.quantum.ibm.com/api/qiskit/circuit_library#particular-quantum-circuits) na documentação da API da biblioteca de circuitos.
## Circuitos aritméticos
Operações aritméticas são funções clássicas, como adição de inteiros e operações bit a bit. Estas podem ser úteis com algoritmos como estimativa de amplitude para aplicações financeiras, e em algoritmos como o algoritmo HHL, que resolve sistemas lineares de equações.

Como exemplo, vamos tentar somar dois números de três bits usando um circuito de "ripple-carry" para realizar adição in-place (`FullAdderGate`). Esse somador adiciona dois números (que chamaremos de "A" e "B") e escreve o resultado no registrador que continha B. No exemplo a seguir, A=2 e B=3.

In [9]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([circuit]).result()

print(f"Count data:\n {result[0].data.c0.get_int_counts()}")

Count data:
 {5: 1024}


![Output of the previous code cell](../docs/images/guides/circuit-library/extracted-outputs/77555a5a-a81c-47b8-a9ae-3015d84adcf5-0.svg)

A simulação do circuito mostra que ele produz `5` em todas as `1024` execuções (ou seja, é medido com probabilidade `1.0`).